# EDA — CSE-CIC-IDS2018 (Zeek conn.log features)

Exploratory Data Analysis of labeled Zeek conn.log flows.

**Data:** `/data/cicids_zeek/<day>/labeled.csv`  
**Features:** ~15 Zeek conn.log fields  
**Labels:** BENIGN + attack classes from UNB schedule

In [1]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless server

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('/data/cicids_zeek')

# Только дни с сетевыми (L3/L4) атаками
DAYS = [
    'Wednesday-14-02-2018',   # FTP-BruteForce, SSH-Bruteforce
    'Thursday-15-02-2018',    # DoS-GoldenEye, DoS-Slowloris
    'Friday-16-02-2018',      # DoS-Hulk, DoS-SlowHTTPTest
    'Wednesday-21-02-2018',   # DDOS-HOIC
    'Friday-02-03-2018',      # Bot
]

NUM_FEATURES = [
    'duration', 'orig_bytes', 'resp_bytes',
    'orig_pkts', 'resp_pkts',
    'orig_ip_bytes', 'resp_ip_bytes'
]
CAT_FEATURES = ['proto', 'service', 'conn_state', 'history']

print('Setup done')
print(f'Days: {len(DAYS)}')
print(f'Features: {len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical')

Setup done
Days: 5
Features: 7 numeric + 4 categorical


## 1. Загрузка данных

In [2]:
dfs = []
for day in DAYS:
    path = DATA_DIR / day / 'labeled.csv'
    if not path.exists():
        print(f'[skip] {day}')
        continue
    df = pd.read_csv(path, low_memory=False)
    df['day'] = day
    dfs.append(df)
    print(f'{day}: {len(df):>10,} rows | attacks: {(df["label"] != "BENIGN").sum():>8,}')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nTotal:   {len(df_all):,} flows')
print(f'Attacks: {(df_all["label"] != "BENIGN").sum():,} ({(df_all["label"] != "BENIGN").mean()*100:.1f}%)')
print(f'Columns: {list(df_all.columns)}')

Wednesday-14-02-2018:  5,551,501 rows | attacks:  282,948
Thursday-15-02-2018:  5,112,131 rows | attacks:   35,640
Friday-16-02-2018:  7,054,949 rows | attacks: 1,909,235
Wednesday-21-02-2018:  6,838,928 rows | attacks: 1,074,408
Friday-02-03-2018:  6,252,667 rows | attacks:   48,007

Total:   30,810,176 flows
Attacks: 3,350,238 (10.9%)
Columns: ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes', 'history', 'label', 'day']


## 2. Распределение классов

In [3]:
label_counts = df_all['label'].value_counts()
print('=== Class distribution ===')
for label, count in label_counts.items():
    pct = count / len(df_all) * 100
    print(f'  {label:<40} {count:>10,}  ({pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Все классы
label_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class distribution (all)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Только атаки (без BENIGN)
attacks = label_counts[label_counts.index != 'BENIGN']
attacks.plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Attack classes only')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/tmp/class_distribution.png', bbox_inches='tight')
plt.show()

=== Class distribution ===
  BENIGN                                   27,459,938  (89.13%)
  DoS-Hulk                                  1,803,685  (5.85%)
  DDOS-HOIC                                 1,074,408  (3.49%)
  FTP-BruteForce                              190,300  (0.62%)
  DoS-SlowHTTPTest                            105,550  (0.34%)
  SSH-Bruteforce                               92,648  (0.30%)
  Bot                                          48,007  (0.16%)
  DoS-GoldenEye                                28,661  (0.09%)
  DoS-Slowloris                                 6,979  (0.02%)


## 3. Распределение по дням

In [4]:
day_label = df_all.groupby(['day', 'label']).size().unstack(fill_value=0)

# Только дни с атаками
attack_cols = [c for c in day_label.columns if c != 'BENIGN']
day_attacks = day_label[attack_cols]
day_attacks = day_attacks[day_attacks.sum(axis=1) > 0]

print('Attack flows per day:')
print(day_attacks.to_string())

day_attacks.plot(kind='bar', stacked=True, figsize=(14, 5), colormap='tab10')
plt.title('Attack flows by day')
plt.xlabel('')
plt.xticks(rotation=30)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('/tmp/attacks_by_day.png', bbox_inches='tight')
plt.show()

Attack flows per day:
label                   Bot  DDOS-HOIC  DoS-GoldenEye  DoS-Hulk  DoS-SlowHTTPTest  DoS-Slowloris  FTP-BruteForce  SSH-Bruteforce
day                                                                                                                             
Friday-02-03-2018     48007          0              0         0                 0              0               0               0
Friday-16-02-2018         0          0              0   1803685            105550              0               0               0
Thursday-15-02-2018       0          0          28661         0                 0           6979               0               0
Wednesday-14-02-2018      0          0              0         0                 0              0          190300           92648
Wednesday-21-02-2018      0    1074408              0         0                 0              0               0               0


## 4. Предобработка признаков

In [5]:
# Заменить '-' на NaN, привести к числу
for col in NUM_FEATURES:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col].replace('-', np.nan), errors='coerce')

# Заполнить NaN нулями для числовых
df_all[NUM_FEATURES] = df_all[NUM_FEATURES].fillna(0)

print('Missing values after fill:')
print(df_all[NUM_FEATURES].isnull().sum())
print()
print('Numeric features stats:')
df_all[NUM_FEATURES].describe().map(lambda x: f'{x:.2f}')

Missing values after fill:
duration         0
orig_bytes       0
resp_bytes       0
orig_pkts        0
resp_pkts        0
orig_ip_bytes    0
resp_ip_bytes    0
dtype: int64

Numeric features stats:


,duration,orig_bytes,resp_bytes,orig_pkts,resp_pkts,orig_ip_bytes,resp_ip_bytes
count,30810176.00,30810176.00,30810176.00,30810176.00,30810176.00,30810176.00,30810176.00
mean,17.91,1563.19,8627.51,13.49,8.73,1155.44,6166.69
std,179.84,968664.95,3483635.73,3701.80,333.45,222157.00,492780.90
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,39.00,70.00,1.00,1.00,69.00,113.00
50%,0.08,97.00,204.00,4.00,3.00,276.00,341.00
75%,2.34,777.00,1581.00,8.00,7.00,1364.00,1872.00
max,44281.05,3512958958.00,12884901887.00,2669391.00,211457.00,160163460.00,315757037.00


## 5. Feature distributions по классам атак

In [6]:
# Топ-5 атак по количеству flows
top_attacks = df_all[df_all['label'] != 'BENIGN']['label'].value_counts().head(5).index.tolist()
plot_labels = ['BENIGN'] + top_attacks
df_plot = df_all[df_all['label'].isin(plot_labels)].copy()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(NUM_FEATURES):
    ax = axes[i]
    for label in plot_labels:
        vals = df_plot[df_plot['label'] == label][feat]
        # Log scale — добавить 1 чтобы избежать log(0)
        vals_log = np.log1p(vals.clip(lower=0))
        vals_log.hist(ax=ax, alpha=0.5, bins=50, label=label, density=True)
    ax.set_title(feat)
    ax.set_xlabel('log1p(value)')
    ax.legend(fontsize=7)

# Скрыть лишний subplot
axes[-1].set_visible(False)

plt.suptitle('Feature distributions by class (log1p scale)', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/feature_distributions.png', bbox_inches='tight')
plt.show()

## 6. Категориальные признаки

In [ ]:
for x in ['proto', 'conn_state', 'service']:
    if feat not in df_all.columns:
        continue
    print(f'\n=== {x} ===')
    ct = pd.crosstab(df_all[x], df_all['label'], normalize='index') * 100
    # Показать только топ значения
    top_vals = df_all[x].value_counts().head(10).index
    print(ct.loc[ct.index.isin(top_vals)].round(1).to_string())

: 

## 7. Корреляционная матрица

In [8]:
# Выборка для корреляции (большой датасет — сэмплируем)
sample = df_all.sample(min(100_000, len(df_all)), random_state=42)

corr = sample[NUM_FEATURES].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.savefig('/tmp/correlation_matrix.png', bbox_inches='tight')
plt.show()

## 8. Class imbalance — подготовка к ML

In [9]:
imbalance = df_all['label'].value_counts()
majority = imbalance.max()

print('Imbalance ratios (vs majority class):')
for label, count in imbalance.items():
    ratio = majority / count
    print(f'  {label:<40} {count:>10,}  ratio: 1:{ratio:.0f}')

print()
print('Recommendation:')
benign_count = imbalance.get('BENIGN', 0)
attack_total = imbalance[imbalance.index != 'BENIGN'].sum()
print(f'  BENIGN:  {benign_count:,} ({benign_count/len(df_all)*100:.1f}%)')
print(f'  Attacks: {attack_total:,} ({attack_total/len(df_all)*100:.1f}%)')
print(f'  → Use class_weight="balanced" or SMOTE for minority classes')

Imbalance ratios (vs majority class):
  BENIGN                                   27,459,938  ratio: 1:1
  DoS-Hulk                                  1,803,685  ratio: 1:15
  DDOS-HOIC                                 1,074,408  ratio: 1:26
  FTP-BruteForce                              190,300  ratio: 1:144
  DoS-SlowHTTPTest                            105,550  ratio: 1:260
  SSH-Bruteforce                               92,648  ratio: 1:296
  Bot                                          48,007  ratio: 1:572
  DoS-GoldenEye                                28,661  ratio: 1:958
  DoS-Slowloris                                 6,979  ratio: 1:3935

Recommendation:
  BENIGN:  27,459,938 (89.1%)
  Attacks: 3,350,238 (10.9%)
  → Use class_weight="balanced" or SMOTE for minority classes


## 9. Сохранение финального датасета для ML

In [ ]:
#подготовка датасета для ML
#категориальные признаки сохраняются как RAW STRINGS (не кодировать здесь!)
#кодирование делают train_*.py скрипты через LabelEncoder при обучении.
#энкодеры сохраняются через save_cat_encoders.py - используются consumer.py.
#ноутбук не должен перезаписывать ml_dataset.parquet — для этого есть make_dataset.py.

df_ml = df_all[NUM_FEATURES + CAT_FEATURES + ['label']].copy()

# Категориальные — оставить как строки, только заполнить пропуски
for col in CAT_FEATURES:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].fillna('-').astype(str)

# Бинарная метка для EDA: 0=BENIGN, 1=attack
df_ml['label_binary'] = (df_ml['label'] != 'BENIGN').astype(int)

#EDA: посмотреть распределение, не сохранять parquet отсюда
#для создания ml_dataset.parquet
#python3 scripts/make_dataset.py
print(f'Shape: {df_ml.shape}')
print(f'Features: {NUM_FEATURES + CAT_FEATURES}')
print(f'\nLabel distribution:')
print(df_ml['label'].value_counts())
print(f'\nProto (должны быть строки): {df_ml["proto"].unique()[:5]}')

# если нужно сохранить parquet вручную:
# OUT = '/data/cicids_zeek/ml_dataset.parquet'
# df_ml.to_parquet(OUT, index=False)


Saved: /data/cicids_zeek/ml_dataset.parquet
Shape: (30810176, 13)
Features: ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes', 'proto', 'service', 'conn_state', 'history']

Label distribution:
label
BENIGN              27459938
DoS-Hulk             1803685
DDOS-HOIC            1074408
FTP-BruteForce        190300
DoS-SlowHTTPTest      105550
SSH-Bruteforce         92648
Bot                    48007
DoS-GoldenEye          28661
DoS-Slowloris           6979
Name: count, dtype: int64
